# 레슨 10 — 통합 프로젝트: 자동화 리포트 만들기

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/10/%5B%ED%95%99%EC%83%9D%EC%9A%A9%5D%20%EB%A0%88%EC%8A%A8%2010%20%E2%80%94%20%ED%86%B5%ED%95%A9%20%ED%94%84%EB%A1%9C%EC%A0%9D%ED%8A%B8%3A%20%EC%9E%90%EB%8F%99%ED%99%94%20%EB%A6%AC%ED%8F%AC%ED%8A%B8%20%EB%A7%8C%EB%93%A4%EA%B8%B0.ipynb)

> 코랩에서 실행하려면 우측 상단 또는 아래의 **Open in Colab** 버튼을 클릭한다. 이 노트북은 학생용 읽기와 따라하기 자료다.


이 노트북은 읽기와 따라하기용 강의 노트북이다. HTML 파싱, 상대 URL, 자료 목록, 품질 검증, 저장, 로그 요약을 하나의 작은 업무 자동화 리포트로 묶는 최종 프로젝트를 안전한 합성 fixture로 연습한다.

## 학습 목표

1. 여러 HTML fixture를 하나의 포털 구조로 해석한다.
2. 링크, 공지, 과정, 다운로드 자료를 각각 추출한다.
3. 중복과 상태 오류를 점검해 저장 전 품질을 확인한다.
4. CSV, JSON, SQLite 산출물을 함께 만든다.
5. 운영자가 읽을 수 있는 3문장 자동화 메모를 작성한다.

---

## 1. 수업 맥락과 안전 기준

마지막 레슨은 “한 페이지에서 값을 뽑았다”가 아니라 “업무 담당자가 바로 확인할 수 있는 리포트”까지 만든다. 합성 포털을 대상으로 하므로 외부 사이트 부하 없이 실제 운영 흐름을 통합 연습한다.

자동화는 빠르게 반복하는 도구이기 때문에 실패했을 때 더 위험해질 수 있다. 그래서 이번 레슨에서는 모든 입력을 수업용 파일로 고정하고, 결과를 저장하기 전에 검증하거나 로그를 남기는 과정을 코드에 포함한다. 이 습관은 실제 사이트를 대상으로 할 때 요청량을 줄이고, 오류를 빨리 발견하게 만든다.

## 2. 환경 셀


In [ ]:
import os
import re
import csv
import json
import time
import sqlite3
import logging
from pathlib import Path
from urllib.parse import urljoin, urlparse

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/10/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_csv(filename):
    text = load_text(filename)
    return list(csv.DictReader(text.splitlines()))

def load_json(filename):
    return json.loads(load_text(filename))

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)) or '0')

def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

def parse_html(filename):
    return BeautifulSoup(load_text(filename), 'html.parser')

def text_or_empty(el):
    return '' if el is None else el.get_text(' ', strip=True)

def status_ok(status):
    return str(status).strip().lower() in {'open', 'ready', 'published'}

def make_key(*parts):
    return '::'.join(str(part).strip().lower() for part in parts)


---

## 3. 핵심 개념

이 셀은 포털 시작 페이지의 제목을 읽으며 프로젝트 입력을 확인한다. 학생은 시작점이 무엇인지 명확히 잡은 뒤 하위 페이지로 이동한다.


In [ ]:
index = parse_html('portal_index.html')
print(text_or_empty(index.select_one('h1')))


시작 페이지 확인은 통합 프로젝트의 기준점을 만든다.

---

## 4. 자료 구조 확인

목차 링크 수집은 사이트맵을 만드는 첫 단계다. label과 href를 함께 저장하면 다음 처리 순서를 사람이 읽을 수 있다.


In [ ]:
links = []
for a in index.select('a[data-page]'):
    links.append({'label': a.get_text(' ', strip=True), 'href': a['href'], 'url': urljoin('https://lesson.local/portal/', a['href'])})
print(links)


링크 목록은 이후 처리할 작업 큐 역할을 한다.

---

## 5. 품질 기준 적용

상대 URL을 절대 URL로 바꾸는 과정은 링크 자동화의 기본이다. 이 원칙은 다운로드 자료와 상세 페이지 수집에도 그대로 이어진다.


In [ ]:
notice = parse_html('portal_notice.html')
notices = [{'title': item.select_one('.title').get_text(' ', strip=True), 'level': item.get('data-level'), 'date': item.get('data-date')} for item in notice.select('.notice-card')]
print(notices[:2])


절대 URL 변환은 다른 페이지로 이동할 때 경로 오류를 줄인다.

---

## 6. 저장과 보고

공지 카드는 반복 단위가 명확한 HTML 구조다. 학생은 카드 개수를 먼저 확인한 뒤 제목, 레벨, 날짜를 분리한다.


In [ ]:
courses = parse_html('portal_courses.html')
course_rows = []
for row in courses.select('tbody tr'):
    cells = [td.get_text(' ', strip=True) for td in row.select('td')]
    course_rows.append({'course': cells[0], 'teacher': cells[1], 'students': clean_int(cells[2]), 'status': cells[3]})
print(course_rows[:2])


카드 개수 확인은 selector가 맞는지 검증하는 빠른 방법이다.

---

## 7. 운영 관점 점검

과정 표는 셀 순서를 읽어 딕셔너리로 바꾸는 연습이다. 학생 수 문자열을 숫자로 정리해야 이후 필터링이 가능하다.


In [ ]:
manifest = load_csv('download_manifest.csv')
print(manifest[:2])


표 행 정리는 과정별 요약을 만드는 기반이다.

---

## 8. 마무리 체크

manifest CSV는 화면 자료와 저장 기준을 연결한다. 파일명과 course를 key로 삼으면 과정별 자료 개수를 안정적으로 계산할 수 있다.


In [ ]:
rules = load_json('quality_rules.json')
ready_courses = [row for row in course_rows if status_ok(row['status']) and row['students'] >= rules['min_students_for_report']]
print(ready_courses)


manifest는 자료 누락 여부를 확인하는 기준 파일이다.

---

## 9. 핵심 개념

품질 규칙 JSON은 리포트 대상 과정을 고르는 기준이다. 코드 안에 숫자를 직접 박지 않고 설정 파일로 분리하는 이유를 설명한다.


In [ ]:
report = {'notice_count': len(notices), 'course_count': len(course_rows), 'file_count': len(manifest), 'ready_course_count': len(ready_courses)}
Path('lesson10_portal_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(report)


설정 JSON은 리포트 조건을 코드 밖으로 분리한다.

---

## 데이터 출처와 안전 규칙

portal_index.html은 합성 포털의 시작 페이지다. portal_notice.html, portal_courses.html, portal_downloads.html, portal_status.html은 각각 공지, 과정, 자료, 상태 정보를 담는다. download_manifest.csv와 quality_rules.json은 최종 리포트 검증에 사용한다.

- 모든 파일은 수업용 합성 데이터다.
- 실제 사이트에 반복 요청하지 않는다.
- 저장 파일은 레슨 폴더 또는 코랩 현재 작업 폴더에만 만든다.
- 외부 사이트로 확장할 때는 약관, robots.txt, 요청 간격, 개인정보 포함 여부를 먼저 확인한다.

---


## 수업 운영 메모

이 절은 학생에게 그대로 읽히는 보충 설명이다. 마지막 레슨은 새 문법을 더 넣는 시간이 아니라 지금까지 배운 파싱, 링크 정리, 검증, 저장, 보고를 하나의 흐름으로 묶는 시간이다. 학생이 셀을 실행할 때마다 “어떤 입력을 읽었고, 어떤 기준으로 걸렀고, 어떤 산출물이 남았는지”를 말하게 하면 프로젝트가 단순 복사 작업으로 흐르지 않는다.

### 1. 통합 프로젝트의 기준

업무 자동화 리포트는 값 몇 개를 출력하는 노트북과 다르다. 시작 페이지에서 하위 페이지를 찾고, 공지와 과정 표와 다운로드 manifest와 상태 metric을 각각 읽은 뒤, 운영자가 확인할 수 있는 CSV와 JSON을 만든다. 따라서 학생은 코드의 길이보다 단계 이름을 분명히 잡아야 한다. 입력 확인, 반복 단위 선택, 품질 기준 적용, 저장, 요약이라는 순서가 무너지면 결과가 맞아 보여도 다시 실행하기 어렵다.

### 2. 포털 링크 수집

portal_index.html의 a[data-page] 링크는 작은 사이트맵이다. 여기서 label, href, 절대 URL을 함께 남기는 이유는 다음 처리 계획을 사람이 읽을 수 있게 하기 위해서다. href만 있으면 어느 페이지인지 알기 어렵고, label만 있으면 실제 이동 경로를 만들 수 없다. 통합 프로젝트에서는 두 값을 같이 저장하는 습관이 중요하다.

### 3. 공지와 과정 데이터 결합

공지 카드는 .notice-card, 과정 표는 tbody tr을 반복 단위로 사용한다. 구조가 다르지만 최종 리포트에서는 같은 운영 상황을 설명하는 재료가 된다. 학생에게 먼저 “반복 단위가 무엇인가”를 묻고, 그 다음 “최종 리포트에 남겨야 할 필드는 무엇인가”를 묻게 한다. 이 순서가 잡히면 table, card, list가 섞여도 당황하지 않는다.

### 4. manifest와 자료 검증

download_manifest.csv는 화면에 보이는 자료가 저장 기준과 맞는지 비교하는 기준 파일이다. 실제 운영에서는 파일명이 바뀌거나 자료가 누락되는 일이 자주 생긴다. 그래서 과정명과 파일명을 묶어 중복 키를 만들고, 과정별 자료 개수를 계산한다. 이 단계는 단순 집계가 아니라 저장 전 품질 검증이다.

### 5. quality_rules 적용

quality_rules.json은 리포트 대상 과정을 고르는 조건을 코드 밖으로 분리한다. 최소 학생 수 같은 기준을 코드에 직접 박으면 다음 주에 기준이 바뀔 때 함수 전체를 고쳐야 한다. 설정 파일에서 기준을 읽고 status_ok()와 함께 적용하면 운영자가 기준을 조정하기 쉬운 구조가 된다.

### 6. 저장 산출물의 역할

CSV는 사람이 표로 확인하기 쉽고, JSON은 요약 값을 다른 코드에서 다시 읽기 쉽고, SQLite는 쿼리로 조회하기 쉽다. 세 가지를 모두 매번 만들 필요는 없지만, 어떤 상황에 어떤 저장 형식이 맞는지 구분할 수 있어야 한다. 이번 레슨에서는 적어도 CSV와 JSON을 만들고, SQLite는 보너스 또는 심화 확인으로 다룬다.

### 7. 운영 메모 작성

마지막 3문장 요약은 장식이 아니다. 자동화 결과를 학원 운영자나 다음 수업 담당자가 바로 이해하게 만드는 전달 문장이다. 좋은 요약은 입력 개수, 리포트 대상 개수, 실행 또는 오류 metric을 포함한다. “잘 됐다”보다 “공지 4건, 과정 6건, 리포트 대상 4건, 오류 1건”처럼 확인 가능한 숫자를 넣는 편이 좋다.

### 8. 수업 중 확인 질문

- 시작 페이지에서 어떤 하위 페이지를 찾았는가?
- 공지 카드와 과정 표의 반복 단위는 각각 무엇인가?
- 리포트 대상 과정은 어떤 기준으로 골랐는가?
- 다운로드 manifest에서 중복 키를 만든 이유는 무엇인가?
- CSV와 JSON 중 어떤 파일이 사람에게 읽기 편한가?
- 최종 요약에 반드시 들어가야 할 숫자는 무엇인가?

### 9. 실제 사이트 확장 전 기준

이 프로젝트는 합성 fixture에서만 실행한다. 실제 사이트로 확장할 때는 약관, robots.txt, 개인정보 포함 여부, 요청 간격, 저장 위치를 먼저 확인한다. 특히 통합 프로젝트는 여러 페이지를 순서대로 읽기 때문에 작은 실수가 반복 요청으로 커질 수 있다. 수업에서는 빠른 자동화보다 안전한 자동화가 목표라는 점을 계속 강조한다.


### 10. 통합 리포트 설계 순서

최종 프로젝트에서는 셀을 빠르게 실행하는 것보다 리포트의 설계 순서를 먼저 잡는 편이 중요하다. 학생에게 아래 순서를 노트북 상단에 적게 하고, 각 단계가 끝날 때마다 출력으로 확인하게 한다.

1. 시작점 확인: 포털 제목과 링크 개수를 확인한다.
2. 입력별 반복 단위 확인: 공지 card, 과정 table row, manifest row, metric item을 구분한다.
3. 변환 기준 적용: 학생 수는 숫자로, 상태는 허용 상태로, 파일은 과정명과 파일명으로 묶는다.
4. 검증: 중복 키, 자료 개수, 리포트 대상 개수를 확인한다.
5. 저장: CSV에는 행 목록, JSON에는 요약 숫자를 저장한다.
6. 보고: 운영자가 읽을 수 있는 3문장 메모를 작성한다.

이 순서가 잡혀 있으면 학생이 어느 셀에서 막혔는지 빠르게 찾을 수 있다. 예를 들어 CSV 저장에서 오류가 난 학생에게 바로 저장 코드를 고치게 하지 말고, report_rows가 정상적으로 만들어졌는지 먼저 보게 한다. report_rows가 비어 있다면 저장 문제가 아니라 필터 기준이나 manifest 결합 문제다.

### 11. 오류를 찾는 순서

통합 프로젝트에서 가장 흔한 오류는 파일명 오타, selector 오타, 이전 변수 이름 불일치다. 학생에게 오류 메시지를 볼 때 아래 순서로 점검하게 한다.

- FileNotFoundError: DATA_BASE와 파일명을 확인한다.
- AttributeError: select_one 결과가 None인지 확인한다.
- KeyError: CSV 헤더 또는 JSON 키를 먼저 출력한다.
- TypeError: 문자열 숫자 변환이 빠졌는지 확인한다.
- 빈 리스트: selector가 너무 좁거나 필터 조건이 너무 강한지 확인한다.

이 순서를 알려주면 학생이 에러가 날 때마다 정답 코드를 찾으려 하지 않고 자기 코드의 입력 상태를 점검하게 된다. 특히 마지막 프로젝트는 여러 입력이 연결되기 때문에, 에러가 난 셀보다 앞 셀의 출력이 원인인 경우가 많다.

### 12. 리포트 행을 작게 유지하는 이유

report_rows에는 운영자가 바로 판단할 수 있는 필드만 넣는다. 과정명, 담당자, 학생 수, 자료 개수, 상태 정도면 충분하다. 모든 공지 제목과 모든 metric을 한 행에 넣으면 CSV가 넓어지고 다음 사람이 읽기 어렵다. 상세 데이터는 별도 CSV로 분리하고, 최종 리포트는 핵심 요약만 담는 방식이 좋다.

학생이 많은 필드를 넣고 싶어 하면 “이 컬럼을 보고 운영자가 어떤 행동을 할 수 있는가?”라고 묻는다. 행동으로 이어지지 않는 컬럼은 최종 리포트가 아니라 디버깅 출력에 가깝다.

### 13. 저장 파일 확인 습관

파일을 저장했다는 출력만으로는 부족하다. 저장 직후 Path.exists(), 행 수, 첫 행 일부를 함께 확인하게 한다. 코랩에서는 작업 폴더가 로컬과 다르기 때문에 파일이 만들어졌는지 직접 확인하는 습관이 중요하다. 이 레슨에서는 CSV와 JSON이 생성된 뒤 summary 숫자와 report_rows 길이가 서로 맞는지 비교한다.

좋은 최종 출력 예시는 다음과 같다.

- report_rows: 4건
- lesson10_portal_report.csv 저장 완료
- lesson10_portal_report.json 저장 완료
- 공지 4건, 과정 6건, 리포트 대상 4건, 오류 1건

이 정도면 강사가 노트북 전체를 다시 읽지 않아도 프로젝트가 어떤 결과를 만들었는지 빠르게 파악할 수 있다.

### 14. 수업 마무리 발표 기준

학생 발표는 코드 줄 설명이 아니라 자동화 흐름 설명이어야 한다. 발표 기준은 다음 세 문장으로 제한한다.

1. “저는 포털 시작 페이지, 공지, 과정, 자료 manifest, 상태 metric을 읽었습니다.”
2. “리포트 대상은 상태가 허용되고 학생 수 기준을 넘는 과정으로 골랐습니다.”
3. “CSV와 JSON을 저장했고, 다음에는 오류 항목만 따로 모으는 기능을 추가할 수 있습니다.”

이 발표가 가능하면 학생은 코드의 핵심 구조를 이해한 것이다. 반대로 출력값만 읽고 끝나는 발표는 자동화의 운영 목적을 놓친 것이므로 다시 입력, 검증, 저장 순서로 설명하게 한다.


### 15. 강의 중 미니 리뷰

각 주요 셀을 실행한 뒤 30초 미니 리뷰를 넣는다. 포털 제목 셀에서는 “시작 파일이 맞는가”, 링크 수집 셀에서는 “하위 페이지가 몇 개인가”, 과정 표 셀에서는 “학생 수가 숫자로 바뀌었는가”, 저장 셀에서는 “파일이 실제 생성되었는가”를 확인한다. 이 짧은 리뷰가 없으면 학생은 셀을 모두 실행했는데도 최종 프로젝트 구조를 설명하지 못할 수 있다.

마지막 수업의 목표는 완성 코드를 많이 쓰는 것이 아니라, 자동화 결과를 믿을 수 있게 만드는 습관을 갖추는 것이다. 입력을 확인하고, 기준을 적용하고, 저장 전에 검증하고, 결과를 문장으로 남기는 흐름이 다음 프로젝트의 기본형이 된다.
